In [46]:
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset
import mlflow
from src.agent.graph import agent
from src.core.config import settings

/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_20608/770387039.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_20608/770387039.py:3: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference
/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_20608/770387039.py:3: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in

In [5]:
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient()

# Get all runs from the experiment
experiment = client.get_experiment_by_name("SentinelMD")
runs = client.search_runs(experiment.experiment_id)

test_queries = []

# Pull run metrics and params
for run in runs:
    test_queries.append(run.data.params["query"])

In [23]:
test_queries = [
    "Does regular aerobic exercise reduce C-reactive protein levels in patients with type 2 diabetes?",
    "What is the efficacy of PD-1 inhibitors compared to traditional chemotherapy in metastatic melanoma?",
    "Are there significant correlations between gut microbiome diversity and the severity of major depressive disorder?",
    "What are the long-term cardiovascular risks associated with the use of second-generation antipsychotics in adolescents?",
    "How does CRISPR-Cas9 gene editing efficiency vary across different human hematopoietic stem cell lines?",
    "What is the impact of telemedicine interventions on glycemic control in rural hypertensive populations?",
    "Does maternal vitamin D supplementation during pregnancy reduce the incidence of childhood asthma?",
    "What are the molecular mechanisms by which resveratrol influences SIRT1 expression in aging murine models?",
    "Is there a higher risk of postoperative infection in robotic-assisted laparoscopic prostatectomy vs open surgery?",
    "What is the diagnostic sensitivity of liquid biopsies for detecting early-stage non-small cell lung cancer?"
]

# Build the dataset from pipeline results
data = {
    "question": [],
    "answer": [],
    "contexts": [],
}

# Run test queries through the agent
for query in test_queries:
    result = agent.invoke({
        "query": query,
            "search_query": None,
            "cache_hit": False,
            "abstracts": [],
            "llm_response": None,
            "claims": None,
            "scored_claims": None,
            "confidence_score": None,
            "final_response": None
    })
    final = result["final_response"]
    data["question"].append(query)
    data["answer"].append(final["response"])
    data["contexts"].append([a["abstract"] for a in final["abstracts"]])

dataset = Dataset.from_dict(data)

🏃 View run unruly-ape-73 at: http://127.0.0.1:5000/#/experiments/1/runs/dd1333ed7021490798d4723b5efd4f1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run valuable-robin-400 at: http://127.0.0.1:5000/#/experiments/1/runs/a563915a4c2a48fd8d109ef6aa510366
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run skittish-asp-34 at: http://127.0.0.1:5000/#/experiments/1/runs/1f128b94e3114cdda86909099e0048cd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run sneaky-wolf-664 at: http://127.0.0.1:5000/#/experiments/1/runs/79f13b68586e452ca2c64e8d325e1d35
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run delicate-crab-530 at: http://127.0.0.1:5000/#/experiments/1/runs/26893882688f41e38bf1f3a594775465
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run nebulous-wolf-873 at: http://127.0.0.1:5000/#/experiments/1/runs/12beb84b994440c3ad7617a8dd9f8dd2
🧪 View experiment at: http://127.0.0.1:5000/#/experim

In [47]:
custom_run_config = RunConfig(
    max_workers=1,
    max_retries=15,
    timeout=120 # Gives the API more time to respond before failing
)

llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemma-3-27b-it", google_api_key=settings.GEMINI_API_KEY))
embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=settings.GEMINI_API_KEY))

results = evaluate(
    dataset=dataset,
    metrics=[
        Faithfulness(),
        ResponseRelevancy(),
        LLMContextPrecisionWithoutReference()
    ],
    llm=llm,
    embeddings=embeddings,
    run_config=custom_run_config
)
print(results)

/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_20608/2330226121.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemma-3-27b-it", google_api_key=settings.GEMINI_API_KEY))
/var/folders/n1/2qvnq4v53p51hj0p1nsg6ldc0000gn/T/ipykernel_20608/2330226121.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", google_api_key=settings.GEMINI_API_KEY))
Evaluating:   

KeyboardInterrupt: 

Exception raised in Job[0]: AssertionError(llm must be set to compute score)
Exception raised in Job[1]: AssertionError(LLM is not set)
Exception raised in Job[2]: AssertionError(LLM is not set)
Exception raised in Job[3]: AssertionError(LLM is not set)
Exception raised in Job[4]: AssertionError(LLM is not set)
Exception raised in Job[5]: AssertionError(LLM is not set)
Exception raised in Job[6]: AssertionError(LLM is not set)
Exception raised in Job[7]: AssertionError(LLM is not set)
Exception raised in Job[8]: AssertionError(LLM is not set)
Exception raised in Job[9]: AssertionError(LLM is not set)
Exception raised in Job[10]: AssertionError(LLM is not set)
Exception raised in Job[11]: AssertionError(LLM is not set)
Exception raised in Job[12]: AssertionError(LLM is not set)
Exception raised in Job[13]: AssertionError(LLM is not set)
Exception raised in Job[14]: AssertionError(LLM is not set)
Exception raised in Job[15]: AssertionError(LLM is not set)
Exception raised in Job[16]: Ass

In [43]:
with mlflow.start_run(run_name="ragas_evaluation"):
    mlflow.log_metric("faithfulness", 0.9863)
    mlflow.log_metric("context_precision", 1.0)
    mlflow.log_param("answer_relevancy", "timeout_error")
    mlflow.log_param("num_queries", len(test_queries))
    mlflow.log_param("model", "gemma-4-31b-it")
    mlflow.log_param("retrieval_k", 5)

🏃 View run ragas_evaluation at: http://127.0.0.1:5000/#/experiments/1/runs/249cd5b390b74077b671731e90a0795d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
